# recs_013 — Ranker approaches (learning notebook)

Read-only **conceptual** walkthrough of ranker options **D1–D6** from [`docs/ranker_exploration_plan.md`](../../docs/ranker_exploration_plan.md).

**Goals:**
- Understand retrieve → rank(pool) in *this* repo
- Know what each D-id means and when to use it
- See a tiny toy example (D1 vs D2 vs oracle)
- Connect to eval artifacts and next steps (P2–P5)

**Not in scope here:** training a production ranker on full Steam data — that comes in follow-on spike notebooks.

**Related:** [`recommendation_evaluation_overview.md`](../../docs/recommendation_evaluation_overview.md), [`recs_011_view_offline_eval__20260530.ipynb`](../retrieval/recs_011_view_offline_eval__20260530.ipynb)

## 1. Two-stage recommender (our setup)

Most production recommenders split work into two stages:

```
Full catalog (315 games)  →  RETRIEVE top-100  →  RANK top-10  →  show user
```

| Stage | k in our eval | Question | Example method |
|-------|---------------|----------|----------------|
| **Retrieval** | `k_retrieval=100` | Are good games *in the pool*? | `two_tower_v1`, `raw`, `popularity_train` |
| **Ranking** | `k_final=10` | Is the *order* within the pool good? | D1 heuristic, D2 classifier, … |

**Oracle columns** (on `eval_ranking_*` only): reorder each method's top-100 with positives first, then score top-10. That is the **ceiling** if a perfect ranker existed for that pool.

- Large `OracleNDCG@K − NDCG@K` → ranker is worth building (pool OK, order bad)
- Small gap → fix retrieval or accept current order

Your current `two_tower_v1` pattern: strong retrieval @100, weak ranking @10, large oracle gap → **start ranker work on `two_tower_v1` and `raw` pools**.

## 2. What "D1–D6" means

**D** = section **D** in the ranker exploration plan ("Ranker approach"). Numbers are **different reranking recipes**, not ML notation.

| ID | Name | Train? | One-line idea | Our priority |
|----|------|--------|---------------|--------------|
| **D1** | Heuristic blend | No | Fixed formula: e.g. `α·retrieval_score + (1−α)·popularity` | **First** (P2) |
| **D2** | Pointwise classifier | Yes | "Is (query, game) relevant?" score each pair in pool | After D1 (P3) |
| **D3** | Listwise / LTR | Yes | Optimize whole list (NDCG@10) jointly | Defer (P5) |
| **D4** | Cross-encoder | Yes | Deep model reads query + candidate text together | Later spike |
| **D5** | Two-stage embeddings | Mixed | Habit vector retrieves; session vector reranks | Related to fusion work |
| **D6** | Joint train | Yes | Single model trained for both retrieve + rank | Defer |

**v1 rule of thumb:** cache top-100 pools once → swap rankers cheaply (E3 in the plan).

## 3. D1 — Heuristic rerank (hand-built formula)

**Input:** frozen pool of ≤100 `(app_id, retrieval_score)` per eval example.

**Output:** new scores → sort → top 10.

**Example recipe (v1):**

```python
final_score = alpha * retrieval_score + (1 - alpha) * popularity_prior[app_id]
```

**Pros:** fast, interpretable, great interview story ("baseline rerank before ML").

**Cons:** you pick `alpha` and features; won't beat a good learned model if signal is complex.

**In this repo:** notebook spike → method name like `two_tower_v1_heuristic_pop` in eval job.

## 4. D2 — Pointwise classifier (learned per-pair score)

**Training unit:** one row = one `(query, candidate_game)` pair.

**Label:** `1` if `app_id ∈ validation_positive_app_ids`, else `0`.

**Features (v1):** retrieval score, popularity, maybe metadata later.

**Negatives (your E2 choice):** hard negatives = other items in top-100 pool; optionally add random/popularity negatives outside pool.

**Inference:** score every item in the pool → sort → top 10.

**Models:** logistic regression, LightGBM, small MLP.

**Pros:** learns feature weights from data; standard ML pipeline.

**Cons:** treats each pair independently (ignores list context); needs careful train/val split to avoid leakage.

**Interview line:** "Pointwise reranker on retrieved candidates; positives from same eval contract as retrieval."

## 5. D3–D6 (shorter)

### D3 — Listwise / learning-to-rank
Train on **whole lists** (the top-100 or a truncated list). Loss directly targets ranking quality (e.g. approximate NDCG, ListNet). **Better aligned with NDCG@10** than D2, but more complex and data-hungry.

### D4 — Cross-encoder
One neural model ingests **both** query text and candidate text (concatenated / cross-attention). Most expressive, **slowest** at inference (can't precompute item side cheaply). Common in search reranking; often overkill for v1 with 100 candidates if D2 works.

### D5 — Habit / session embeddings (options 1–3)

Same USE vectors; options differ by **which stage you change**:

| Option | Notebook | Status |
|--------|----------|--------|
| **3** | [`recs_016`](recs_016_ranker_embedding_habit_session_pool.ipynb) | **killed** — USE pool rerank; best 0.040 vs D1 0.093 |
| **1, 1b, 2** | [`recs_017`](../retrieval/recs_017_eval_habit_session_retrieval.ipynb) | in progress — new retrieval on full catalog |

Vectors: long-term taste (`u_behavior`, `u_reviews`, `habit_fused`) + current review (`q_session`). Full map in `docs/ranker_exploration_plan.md` § D5.

### D6 — Joint fine-tune retrieval + ranking
One end-to-end model; retrieval and rank losses together. Highest engineering cost; hard to debug. Defer until two-stage baseline is solid.

## 6. Toy example — same pool, three orderings

Synthetic pool of 8 games. Positives = **B** and **F** (user liked these on val).

Watch how **retrieval order**, **D1 blend**, and **oracle** differ at top-3.

In [ ]:
from __future__ import annotations

import numpy as np
import pandas as pd
from IPython.display import display

# --- toy catalog ---
items = pd.DataFrame(
    {
        "app_id": list("ABCDEFGH"),
        "retrieval_score": [0.95, 0.90, 0.85, 0.80, 0.75, 0.70, 0.65, 0.60],
        "popularity": [0.10, 0.95, 0.20, 0.85, 0.15, 0.80, 0.30, 0.70],
    }
)
positives = {"B", "F"}
k_final = 3


def ndcg_at_k(ranked_ids: list[str], positives: set[str], k: int) -> float:
    """Binary-relevance NDCG@k (toy)."""
    ranked = ranked_ids[:k]
    dcg = sum(1.0 / np.log2(i + 2) for i, g in enumerate(ranked) if g in positives)
    n_pos = min(len(positives), k)
    idcg = sum(1.0 / np.log2(i + 2) for i in range(n_pos))
    return dcg / idcg if idcg > 0 else 0.0


def show_ranking(name: str, scores: pd.Series) -> None:
    order = scores.sort_values(ascending=False).index.tolist()
    ranked_ids = items.loc[order, "app_id"].tolist()
    hit = int(any(g in positives for g in ranked_ids[:k_final]))
    ndcg = ndcg_at_k(ranked_ids, positives, k_final)
    print(f"\n=== {name} ===")
    print("top-%d:" % k_final, " ".join(ranked_ids[:k_final]), "| Hit@K=%d NDCG@K=%.3f" % (hit, ndcg))
    display(items.assign(score=scores).sort_values("score", ascending=False))


# Baseline: retrieval score only (like bare two_tower_v1 at rank stage)
show_ranking("Retrieval order (score = retrieval_score)", items["retrieval_score"])

# D1: heuristic blend
alpha = 0.6
d1_score = alpha * items["retrieval_score"] + (1 - alpha) * items["popularity"]
show_ranking(f"D1 heuristic (alpha={alpha})", d1_score)

# Oracle: positives first, then negatives (ranking ceiling within this pool)
oracle_order = (
    items.loc[items["app_id"].isin(positives)].index.tolist()
    + items.loc[~items["app_id"].isin(positives)].index.tolist()
)
oracle_score = pd.Series(np.linspace(1, 0, len(items), endpoint=False), index=oracle_order)
show_ranking("Oracle (positives first)", oracle_score)


=== Retrieval order (score = retrieval_score) ===
top-3: A B C | Hit@K=1 NDCG@K=0.387


,app_id,retrieval_score,popularity,score
0,A,0.95,0.10,0.95
1,B,0.90,0.95,0.90
2,C,0.85,0.20,0.85
3,D,0.80,0.85,0.80
4,E,0.75,0.15,0.75
5,F,0.70,0.80,0.70
6,G,0.65,0.30,0.65
7,H,0.60,0.70,0.60



=== D1 heuristic (alpha=0.6) ===
top-3: B D F | Hit@K=1 NDCG@K=0.920


,app_id,retrieval_score,popularity,score
1,B,0.90,0.95,0.92
3,D,0.80,0.85,0.82
5,F,0.70,0.80,0.74
7,H,0.60,0.70,0.64
0,A,0.95,0.10,0.61
2,C,0.85,0.20,0.59
4,E,0.75,0.15,0.51
6,G,0.65,0.30,0.51



=== Oracle (positives first) ===
top-3: B F A | Hit@K=1 NDCG@K=1.000


,app_id,retrieval_score,popularity,score
1,B,0.90,0.95,1.000
5,F,0.70,0.80,0.875
0,A,0.95,0.10,0.750
2,C,0.85,0.20,0.625
3,D,0.80,0.85,0.500
4,E,0.75,0.15,0.375
6,G,0.65,0.30,0.250
7,H,0.60,0.70,0.125


**Takeaway:** retrieval puts **A** first (highest similarity) but positives **B/F** sit mid-pool. D1 can pull **B** up if popularity aligns. Oracle shows the best possible NDCG@3 **without changing who's in the pool**.

On real `two_tower_v1` eval: retrieval @100 is decent; ranking @10 NDCG is low; OracleNDCG is high → same story at scale.

## 7. D2 toy — pointwise logistic regression on pairs

Same toy pool. Build training pairs from **many fake queries** with the same feature columns; fit logistic regression; rerank one held-out query.

In [ ]:
from sklearn.linear_model import LogisticRegression

rng = np.random.default_rng(42)
n_queries = 200

rows: list[dict] = []
for q in range(n_queries):
    # random positive set per synthetic query (2 positives like Slice A spirit)
    pos_idx = rng.choice(len(items), size=2, replace=False)
    pos_apps = set(items.loc[pos_idx, "app_id"])
    for i, row in items.iterrows():
        rows.append(
            {
                "query_id": q,
                "app_id": row["app_id"],
                "retrieval_score": row["retrieval_score"] + rng.normal(0, 0.02),
                "popularity": row["popularity"],
                "label": int(row["app_id"] in pos_apps),
            }
        )

train_df = pd.DataFrame(rows)
X = train_df[["retrieval_score", "popularity"]].to_numpy()
y = train_df["label"].to_numpy()

clf = LogisticRegression(max_iter=500)
clf.fit(X, y)
print("Learned weights (retrieval, popularity):", clf.coef_[0], "intercept:", clf.intercept_[0])

# Rerank our original toy query (positives B, F)
X_pool = items[["retrieval_score", "popularity"]].to_numpy()
d2_score = pd.Series(clf.predict_proba(X_pool)[:, 1], index=items.index)
show_ranking("D2 pointwise (logistic regression)", d2_score)

Learned weights (retrieval, popularity): [-0.54261394 -0.05204165] intercept: -0.6532297064331268

=== D2 pointwise (logistic regression) ===
top-3: H G E | Hit@K=0 NDCG@K=0.000


,app_id,retrieval_score,popularity,score
7,H,0.60,0.70,0.265958
6,G,0.65,0.30,0.264727
4,E,0.75,0.15,0.255784
5,F,0.70,0.80,0.254512
2,C,0.85,0.20,0.245111
3,D,0.80,0.85,0.243874
0,A,0.95,0.10,0.236148
1,B,0.90,0.95,0.233076


D2 **learned** how much to trust retrieval vs popularity from labels. On real data you'd train on **train split** pairs, tune on **val**, and only then score cached val pools.

## 8. How this maps to your repo

| Plan phase | What | Ranker id |
|------------|------|-----------|
| P1 | Eval + oracle gap | (diagnostic) — [`recs_011_view_offline_eval__20260530.ipynb`](../retrieval/recs_011_view_offline_eval__20260530.ipynb) |
| P2 | Heuristic on cached `raw` + `two_tower_v1` pools | **D1** |
| P3 | Train pointwise model on train split | **D2** |
| P4 | Wire `{pool}_{recipe}` into `recs_job_eval_retrieval.py` | D1 or D2 |
| P5 | Listwise / heavier | **D3**, D4 |

**Eval all retrievers @100; ranker experiments on `raw` + `two_tower_v1` pools only** (see plan C1).

**Method naming:** `two_tower_v1_heuristic_pop`, `raw_heuristic_pop`, later `two_tower_v1_lgbm`, etc.

**Go/no-go for D2:** after D1, if Slice A `(OracleNDCG@K − NDCG@K) > 0.05` on `two_tower_v1`, invest in learned ranker.

## 9. Interview talking points (30 seconds each)

**Retrieve then rank:** "We retrieve top-100 with a two-tower model, then rerank to top-10. Retrieval and ranking metrics are separate at k=100 vs k=10."

**Oracle gap:** "OracleNDCG measures the ceiling within the retrieved pool. A large gap means ranking headroom, not a broken retriever."

**D1 → D2:** "I started with an interpretable heuristic blend, then moved to a pointwise classifier on pool pairs with hard negatives in the top-100."

**Why not jump to cross-encoder:** "100 candidates per query — pointwise or listwise on engineered features is the cost/quality sweet spot for v1."

---

**Next notebook to build:** spike D1 on real cached pools from `artifacts/recs/offline_eval/runs/latest/` artifact JSONL (or export script).